<a href="https://colab.research.google.com/github/cook1e-0707/practicalAI-lab/blob/main/day4/student/week1_day4_yourname.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

# Week 1 · Day 4 Student Lab
## Build and Train a Tiny Language Model

**Before you start:** Choose **File → Save a copy in Drive**.
Rename it `week1_day4_yourname.ipynb`.


## Today

We will follow the smallest complete LLM demonstration:

1. start with food-related text;
2. build a tokenizer;
3. turn text into token IDs;
4. create next-token training pairs;
5. train a tiny GPT-style model;
6. generate new text one token at a time.

<div style="overflow-x:auto; margin:14px 0;">
  <table style="width:100%; min-width:900px; border-collapse:separate; border-spacing:6px;">
    <tr>
      <td style="background:#e8f1ff; border:2px solid #246bce; border-radius:10px; padding:12px; text-align:center;">
        <b>1 · Text</b><br><small>food corpus</small>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#e8f1ff; border:2px solid #246bce; border-radius:10px; padding:12px; text-align:center;">
        <b>2 · Tokenize</b><br><small>split + assign IDs</small>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#e8f1ff; border:2px solid #246bce; border-radius:10px; padding:12px; text-align:center;">
        <b>3 · Shift</b><br><small>input + target</small>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#fff2d8; border:2px solid #b96900; border-radius:10px; padding:12px; text-align:center;">
        <b>4 · Predict</b><br><small>TinyGPT logits</small>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#fff2d8; border:2px solid #b96900; border-radius:10px; padding:12px; text-align:center;">
        <b>5 · Learn</b><br><small>loss + update</small>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#e5f7f2; border:2px solid #138a72; border-radius:10px; padding:12px; text-align:center;">
        <b>6 · Generate</b><br><small>append + repeat</small>
      </td>
    </tr>
  </table>
</div>

`Day4TinyGPT` is a real decoder-only Transformer with about
0.43 million parameters. It demonstrates the process, but it is not
a production LLM and its generated statements are not guaranteed
facts.


### Why We Build a Tiny LLM

| Classroom TinyGPT | Public large-model comparison |
|---|---|
| About **0.43 million parameters** | Llama 3.1 405B has **405 billion parameters** |
| About **348,000 characters** of food text | Llama 3.1 used **more than 15 trillion tokens** |
| One Colab CPU or GPU | More than **16,000 NVIDIA H100 GPUs** |
| A short live training run | **30.84 million H100 GPU-hours** for the 405B model |
| Goal: make every learning step visible | Goal: build a broadly capable assistant-scale model |

Llama 3.1 405B has approximately **950,000 times as many
parameters** as our 426,580-parameter TinyGPT. We reduce the
model size, training text, and training steps so the complete
process can run on one Colab computer during class. TinyGPT
still uses the central pipeline: tokenize, predict the next
token, measure loss, update parameters, and generate.

The exact size, hardware, data, and training compute used for
GPT-4 were not published. Llama 3.1 405B is shown only as a
public example of the scale of a leading chat-capable model.

Scale references:

- [OpenAI GPT-4 Technical Report](https://cdn.openai.com/papers/gpt-4.pdf)
- [Meta Llama 3.1 model card](https://github.com/meta-llama/llama-models/blob/main/models/llama3_1/MODEL_CARD.md)
- [Meta Llama 3.1 announcement](https://ai.meta.com/blog/meta-llama-3-1/)


# Part 1 — Packages

Packages have different jobs in this notebook.

| Package | Why it is here | Importance to the LLM pipeline |
|---|---|---|
| `torch` (PyTorch) | Tensors, TinyGPT layers, loss, optimization, and generation | **Core** |
| `torch.nn` | Ready-made neural-network layers such as embeddings and linear layers | **Core** |
| `torch.nn.functional` | Operations such as softmax and cross-entropy loss | **Core** |
| `tiktoken` | A real subword tokenizer for comparison | Important tokenizer example |
| `numpy` | Positions and values used by our plots | Visualization support |
| `matplotlib` | Draws token, loss, and probability figures | Visualization support |
| `pathlib`, `urllib`, `importlib`, `subprocess` | Loads files and prepares the environment | Setup support, not model learning |

The setup checks for missing packages before using pip:

```text
python -m pip install -q torch numpy matplotlib tiktoken
```

Important names in the next cell:

| Name | Meaning |
|---|---|
| `torch.cuda.is_available()` | Checks for an NVIDIA GPU |
| `torch.backends.mps.is_available()` | Checks for Apple GPU support |
| `torch.device(...)` | Selects where PyTorch calculations run |
| `device` | Stores the selected CPU or GPU |
| `.to(device)` | Moves a tensor or model to the selected device |

A device changes training speed. It does not change the definition
of the model.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "torch": "torch",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "tiktoken": "tiktoken",
}
missing_packages = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]
if missing_packages:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing_packages,
        ]
    )

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import tiktoken
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("tiktoken:", tiktoken.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Selected device:", device)
print(
    "Test tensor, 2 × 3 =",
    int(torch.tensor([2, 3], device=device).prod().item()),
)


# Part 2 — Load the Training Text

The text file contains repeated food-related question-and-answer
records. Repetition helps a very small model learn visible patterns
during a short class.

The loader tries a local repository file first, then GitHub, and
finally a Colab upload.

This is a **data-loading step**, not a model-learning step.

| Name | Important parameter | Purpose |
|---|---|---|
| `Path(...)` | file location | Represents a local path |
| `.read_text(...)` | `encoding="utf-8"` | Reads text without losing characters |
| `urlopen(...)` | `timeout=30` | Tries the GitHub URL for at most 30 seconds |
| `.decode("utf-8")` | text encoding | Converts downloaded bytes into a Python string |
| `corpus_text` | — | Stores the complete training text used by the tokenizer |


In [ ]:
from pathlib import Path
from urllib.request import urlopen

DATA_URL = "https://raw.githubusercontent.com/cook1e-0707/practicalAI-lab/main/day4/data/food_knowledge_corpus.txt"
local_data_candidates = [
    Path("day4/data/food_knowledge_corpus.txt"),
    Path("../data/food_knowledge_corpus.txt"),
    Path("../../day4/data/food_knowledge_corpus.txt"),
]
local_data_path = next(
    (path for path in local_data_candidates if path.exists()),
    None,
)

if local_data_path is not None:
    corpus_text = local_data_path.read_text(encoding="utf-8")
    data_source = f"local file: {local_data_path.resolve()}"
else:
    try:
        corpus_text = (
            urlopen(DATA_URL, timeout=30)
            .read()
            .decode("utf-8")
        )
        data_source = "GitHub raw-data URL"
    except Exception:
        from google.colab import files

        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        corpus_text = uploaded[uploaded_name].decode("utf-8")
        data_source = f"uploaded file: {uploaded_name}"

print("Data source:", data_source)
print("Characters:", f"{len(corpus_text):,}")
print("\nFirst 500 characters:\n")
print(corpus_text[:500])


# Part 3 — Tokenizer and Tokenization

A **tokenizer** is the tool that performs tokenization.

**Tokenization** is the process:

```text
text → tokens → token IDs
```

Our first tokenizer uses one character as one token:

- `char_to_id`: character → integer;
- `id_to_char`: integer → character;
- `encode(text)`: text → list of IDs;
- `decode(ids)`: IDs → text.

Important tokenizer-building tools:

| Name | Input | Output and role |
|---|---|---|
| `set(corpus_text)` | all training text | every different character, with duplicates removed |
| `sorted(...)` | the set of characters | a stable vocabulary order |
| `enumerate(characters)` | ordered vocabulary | `(token_id, character)` pairs |
| `char_to_id` | character key | the corresponding token ID |
| `id_to_char` | token-ID key | the corresponding character |
| `encode(text)` | Python string | list of token IDs sent to TinyGPT |
| `decode(token_ids)` | IDs from the model | readable Python string |
| `vocabulary_size` | number of unique tokens | number of possible next-token choices |

### Tokenization Map

<div style="overflow-x:auto; margin:14px 0;">
  <table style="width:100%; min-width:760px; border-collapse:separate; border-spacing:7px;">
    <tr>
      <td style="background:#f3f4f6; border:2px solid #6b7280; border-radius:10px; padding:13px; text-align:center;">
        <b>Text</b><br><code>food</code>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#e8f1ff; border:2px solid #246bce; border-radius:10px; padding:13px; text-align:center;">
        <b>Split into tokens</b><br><code>f | o | o | d</code>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#fff2d8; border:2px solid #b96900; border-radius:10px; padding:13px; text-align:center;">
        <b>Look up each token</b><br><code>char_to_id</code>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#e5f7f2; border:2px solid #138a72; border-radius:10px; padding:13px; text-align:center;">
        <b>Token IDs</b><br><code>[number, number, number, number]</code>
      </td>
    </tr>
  </table>
</div>

The repeated `o` receives the same ID both times. Token IDs are
fixed labels from the vocabulary; they are not importance scores.


In [ ]:
characters = sorted(set(corpus_text))
vocabulary_size = len(characters)

char_to_id = {
    character: token_id
    for token_id, character in enumerate(characters)
}
id_to_char = {
    token_id: character
    for character, token_id in char_to_id.items()
}

def encode(text):
    return [char_to_id[character] for character in text]

def decode(token_ids):
    return "".join(
        id_to_char[int(token_id)]
        for token_id in token_ids
    )

sample_text = "food"
sample_ids = encode(sample_text)

print("Vocabulary size:", vocabulary_size)
print("Text:", sample_text)
print("Token IDs:", sample_ids)
print("Decoded:", decode(sample_ids))


In [ ]:
figure, axis = plt.subplots(figsize=(11, 5))
unique_sample_characters = list(dict.fromkeys(sample_text))
character_colors = {
    character: plt.cm.Set3(color_index)
    for character, color_index in zip(
        unique_sample_characters,
        np.linspace(0, 1, len(unique_sample_characters)),
    )
}

row_y = {
    "Text": 3.4,
    "Tokens": 2.35,
    "Token IDs": 1.3,
    "Decoded text": 0.25,
}
for label, y_position in row_y.items():
    axis.text(
        -1.35,
        y_position,
        label,
        ha="right",
        va="center",
        fontsize=12,
        fontweight="bold",
    )

axis.text(
    (len(sample_text) - 1) / 2,
    row_y["Text"],
    repr(sample_text),
    ha="center",
    va="center",
    fontsize=14,
)

for position, (character, token_id) in enumerate(
    zip(sample_text, sample_ids)
):
    color = character_colors[character]
    axis.text(
        position,
        row_y["Tokens"],
        character,
        ha="center",
        va="center",
        fontsize=14,
        bbox={
            "boxstyle": "round,pad=0.48",
            "facecolor": color,
            "edgecolor": "#444444",
        },
    )
    axis.text(
        position,
        row_y["Token IDs"],
        str(token_id),
        ha="center",
        va="center",
        fontsize=12,
        bbox={
            "boxstyle": "round,pad=0.42",
            "facecolor": color,
            "edgecolor": "#444444",
        },
    )

axis.text(
    (len(sample_text) - 1) / 2,
    row_y["Decoded text"],
    repr(decode(sample_ids)),
    ha="center",
    va="center",
    fontsize=14,
)

arrow_x = len(sample_text) + 0.15
for upper_row, lower_row in [
    ("Text", "Tokens"),
    ("Tokens", "Token IDs"),
    ("Token IDs", "Decoded text"),
]:
    axis.annotate(
        "",
        xy=(arrow_x, row_y[lower_row] + 0.2),
        xytext=(arrow_x, row_y[upper_row] - 0.2),
        arrowprops={"arrowstyle": "->", "linewidth": 1.8},
    )

axis.text(
    (1 + 2) / 2,
    1.82,
    "same token → same ID",
    ha="center",
    va="center",
    fontsize=10,
    color="#7c3aed",
)
axis.plot(
    [1, 2],
    [1.67, 1.67],
    color="#7c3aed",
    linewidth=2,
)

axis.set_xlim(-2.25, len(sample_text) + 0.65)
axis.set_ylim(-0.35, 3.85)
axis.axis("off")
figure.tight_layout()
plt.show()


## Modern Subword Tokenization

Modern LLMs usually use subword tokens instead of one token per
character.

- `tiktoken.get_encoding("r50k_base")` loads an existing BPE
  tokenizer.
- `.encode(...)` returns subword IDs.
- `.decode(...)` reconstructs the text.

BPE means **Byte Pair Encoding**. Common fragments can become one
token, so the same sentence often needs fewer tokens.

| Name | Important input or parameter | Role |
|---|---|---|
| `tiktoken.get_encoding(...)` | `"r50k_base"` | Loads a named, already-built tokenizer vocabulary |
| `bpe_tokenizer.encode(text)` | a Python string | Produces BPE token IDs |
| `bpe_tokenizer.decode(ids)` | one or more BPE IDs | Reconstructs readable text |

`"r50k_base"` is the tokenizer’s name, not a training setting for
TinyGPT. Our TinyGPT still uses the character tokenizer so every
step stays visible.


In [ ]:
bpe_tokenizer = tiktoken.get_encoding("r50k_base")
comparison_text = "food safety"

character_ids = encode(comparison_text)
bpe_ids = bpe_tokenizer.encode(comparison_text)
bpe_pieces = [
    bpe_tokenizer.decode([token_id])
    for token_id in bpe_ids
]

print("Text:", comparison_text)
print("Character-token count:", len(character_ids))
print("BPE-token count:", len(bpe_ids))
print("BPE pieces and IDs:")
print(list(zip(bpe_pieces, bpe_ids)))

character_pieces = [
    "space" if character == " " else character
    for character in comparison_text
]
bpe_visible_pieces = [
    piece.replace(" ", "[space]")
    for piece in bpe_pieces
]

figure, axes = plt.subplots(
    2,
    1,
    figsize=(12, 4.8),
)
token_rows = [
    (
        axes[0],
        "Character tokenizer",
        character_pieces,
        character_ids,
        "#dbeafe",
    ),
    (
        axes[1],
        "BPE tokenizer",
        bpe_visible_pieces,
        bpe_ids,
        "#dcfce7",
    ),
]

for axis, row_label, pieces, token_ids, color in token_rows:
    for position, (piece, token_id) in enumerate(
        zip(pieces, token_ids)
    ):
        axis.text(
            position,
            0.62,
            piece,
            ha="center",
            va="center",
            fontsize=10,
            bbox={
                "boxstyle": "round,pad=0.42",
                "facecolor": color,
                "edgecolor": "#444444",
            },
        )
        axis.text(
            position,
            0.15,
            str(token_id),
            ha="center",
            va="center",
            fontsize=8,
        )

    axis.set_xlim(-0.7, len(pieces) - 0.3)
    axis.set_ylim(-0.12, 1.05)
    axis.set_title(
        f"{row_label}: {len(pieces)} tokens",
        loc="left",
        fontsize=12,
    )
    axis.axis("off")

figure.tight_layout()
plt.show()


## Task 1 — Encode and Decode

**Input:** a short sentence using characters from the corpus.

**Expected output:** token IDs followed by the original sentence.

<details><summary>Hint 1</summary>

Use `encode(...)`, then `decode(...)`.
</details>

<details><summary>Hint 2</summary>

```python
your_ids = encode(your_text)
decoded_text = decode(your_ids)
```
</details>


In [ ]:
your_text = "Vitamin C is in many foods."
your_ids = None  # TODO: encode your_text
decoded_text = None  # TODO: decode your_ids

print("Token IDs:", your_ids)
print("Decoded:", decoded_text)


# Part 4 — Create Next-Token Training Pairs

The same token sequence supplies inputs and correct answers:

```text
input:   f o o d
target:  o o d [next character]
```

The target is shifted forward by one position.

`torch.tensor(..., dtype=torch.long)` stores whole-number token IDs
in the format expected by the model. The first 90% of tokens are
used for training and the remaining 10% for validation.

| Name or parameter | Meaning in the LLM pipeline |
|---|---|
| `torch.tensor(...)` | Converts the Python list of token IDs into a PyTorch tensor |
| `dtype=torch.long` | Stores integer IDs in the type required by `nn.Embedding` |
| `split_index` | Position separating training tokens from validation tokens |
| `[:split_index]` | Takes the first 90% for weight updates |
| `[split_index:]` | Takes the final 10% for evaluation only |
| `[:-1]` | Removes the last token to create model inputs |
| `[1:]` | Removes the first token to create correct next-token targets |

`training_ids` may update the model. `validation_ids` must never be
passed to the optimizer as training data.


In [ ]:
all_token_ids = torch.tensor(
    encode(corpus_text),
    dtype=torch.long,
)
split_index = int(0.90 * len(all_token_ids))
training_ids = all_token_ids[:split_index]
validation_ids = all_token_ids[split_index:]

shift_text = "food safety"
shift_ids = torch.tensor(
    encode(shift_text),
    dtype=torch.long,
)
example_input = shift_ids[:-1]
example_target = shift_ids[1:]

print("Input: ", repr(decode(example_input)))
print("Target:", repr(decode(example_target)))
print(
    "Training / validation tokens:",
    len(training_ids),
    "/",
    len(validation_ids),
)

figure, axis = plt.subplots(figsize=(13, 4))
input_characters = list(decode(example_input))
target_characters = list(decode(example_target))

for position, (input_character, target_character) in enumerate(
    zip(input_characters, target_characters)
):
    visible_input = (
        "space" if input_character == " " else input_character
    )
    visible_target = (
        "space" if target_character == " " else target_character
    )
    axis.text(
        position,
        1.75,
        f"{visible_input}\nID {int(example_input[position])}",
        ha="center",
        va="center",
        fontsize=10,
        bbox={
            "boxstyle": "round,pad=0.38",
            "facecolor": "#dbeafe",
            "edgecolor": "#2563eb",
        },
    )
    axis.text(
        position,
        0.35,
        f"{visible_target}\nID {int(example_target[position])}",
        ha="center",
        va="center",
        fontsize=10,
        bbox={
            "boxstyle": "round,pad=0.38",
            "facecolor": "#fef3c7",
            "edgecolor": "#d97706",
        },
    )
    axis.annotate(
        "",
        xy=(position, 0.72),
        xytext=(position, 1.38),
        arrowprops={
            "arrowstyle": "->",
            "color": "#555555",
        },
    )

axis.text(
    -0.8,
    1.75,
    "Input",
    ha="right",
    va="center",
    fontsize=12,
    fontweight="bold",
)
axis.text(
    -0.8,
    0.35,
    "Correct next token",
    ha="right",
    va="center",
    fontsize=12,
    fontweight="bold",
)
axis.set_xlim(-2.7, len(example_input) - 0.3)
axis.set_ylim(-0.25, 2.3)
axis.axis("off")
figure.tight_layout()
plt.show()


# Part 5 — TinyGPT

<div style="overflow-x:auto; margin:14px 0;">
  <table style="width:100%; min-width:850px; border-collapse:separate; border-spacing:7px;">
    <tr>
      <td style="background:#e8f1ff; border:2px solid #246bce; border-radius:10px; padding:13px; text-align:center;">
        <b>Token IDs</b><br><small>whole numbers</small>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#e8f1ff; border:2px solid #246bce; border-radius:10px; padding:13px; text-align:center;">
        <b>Embeddings</b><br><small>token meaning + position</small>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#fff2d8; border:2px solid #b96900; border-radius:10px; padding:13px; text-align:center;">
        <b>Causal attention</b><br><small>use earlier positions</small>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#fff2d8; border:2px solid #b96900; border-radius:10px; padding:13px; text-align:center;">
        <b>Feed-forward network</b><br><small>transform each position</small>
      </td>
      <td style="border:0; text-align:center; font-size:22px;">→</td>
      <td style="background:#e5f7f2; border:2px solid #138a72; border-radius:10px; padding:13px; text-align:center;">
        <b>Logits</b><br><small>one score per possible token</small>
      </td>
    </tr>
  </table>
</div>

Core PyTorch building blocks in the provided model:

| Package name | Role inside TinyGPT |
|---|---|
| `nn.Module` | Base class for a trainable PyTorch model or layer |
| `nn.Embedding` | Looks up a learned vector for each token ID or position |
| `nn.Linear` | Changes vectors and produces final vocabulary logits |
| `nn.LayerNorm` | Keeps vector values on a manageable scale |
| `nn.Dropout` | Randomly hides a small fraction of values during training |
| `nn.GELU` | Nonlinear activation in the feed-forward network |
| `nn.ModuleList` | Stores repeated Transformer blocks as model layers |
| `F.softmax(...)` | Converts attention scores into attention weights |
| `F.cross_entropy(...)` | Compares next-token logits with correct target IDs |

The central attention calculation:

| Operation | Why it exists |
|---|---|
| one `nn.Linear` projection | creates query, key, and value vectors |
| `query @ key.transpose(...)` | scores how strongly token positions relate |
| divide by `sqrt(head_size)` | prevents attention scores from becoming too large |
| `torch.tril(...)` | creates the lower-triangular causal mask |
| `.masked_fill(..., -inf)` | removes future positions before softmax |
| `F.softmax(scores, dim=-1)` | makes allowed scores sum to one |
| `attention_weights @ value` | combines information using the learned weights |
| `x + attention_output` | residual connection preserving the earlier representation |

Core classes and functions defined by this notebook:

| Name | Input | Output or job |
|---|---|---|
| `CausalSelfAttention` | token vectors | mixes information from allowed earlier positions |
| `TransformerBlock` | token vectors | applies attention and a feed-forward network |
| `Day4TinyGPT` | token IDs, plus optional targets | returns logits and optional loss |
| `count_parameters(model)` | a PyTorch model | number of trainable stored values |
| `generate_text_ids(...)` | model, starting IDs, generation settings | longer token-ID sequence |
| `@torch.no_grad()` | generation function | prevents gradient storage during prediction |

**Causal** means a position can use earlier tokens but cannot peek
at future target tokens.

In the provided code, `nn.Embedding` stores learned token and
position vectors. Transformer blocks process those vectors, and
the final linear layer produces the logits.

The next cell defines the provided model. You do not need to write
or memorize it. Focus on the named stages above.

The notebook and the prepared checkpoint use the same TinyGPT
definition. This is necessary because saved parameter values can
only be loaded into the model structure they were trained with.


### Choose the TinyGPT Size

The next cell creates one untrained model. These values are
**hyperparameters**: choices made before training begins.

| Parameter or function | Value here | What it controls |
|---|---:|---|
| `vocabulary_size` | built from the corpus | number of possible token IDs and output logits |
| `context_length` | `64` | maximum recent tokens used for one prediction |
| `embedding_size` | `128` | number of learned values representing each token position |
| `number_of_heads` | `4` | parallel attention views inside each block |
| `number_of_blocks` | `2` | repeated Transformer blocks |
| `dropout` | `0.1` | fraction hidden randomly during training |
| `torch.manual_seed(42)` | `42` | makes random initialization repeatable |
| `**MODEL_CONFIG` | configuration dictionary | passes all five named settings into `Day4TinyGPT` |
| `.to(device)` | selected CPU or GPU | moves the model to the calculation device |

A larger configuration can learn more complex patterns, but it
needs more memory and training time.


In [ ]:
from __future__ import annotations

import math

import torch
from torch import nn
import torch.nn.functional as F


MODEL_CONFIG = {
    "context_length": 64,
    "embedding_size": 128,
    "number_of_heads": 4,
    "number_of_blocks": 2,
    "dropout": 0.1,
}


class CausalSelfAttention(nn.Module):
    """Let every token use only itself and earlier tokens."""

    def __init__(
        self,
        embedding_size: int,
        number_of_heads: int,
        context_length: int,
        dropout: float,
    ) -> None:
        super().__init__()
        if embedding_size % number_of_heads != 0:
            raise ValueError("embedding_size must divide evenly by heads")
        self.number_of_heads = number_of_heads
        self.head_size = embedding_size // number_of_heads
        self.query_key_value = nn.Linear(
            embedding_size,
            3 * embedding_size,
        )
        self.output_projection = nn.Linear(
            embedding_size,
            embedding_size,
        )
        self.attention_dropout = nn.Dropout(dropout)
        self.output_dropout = nn.Dropout(dropout)
        mask = torch.tril(
            torch.ones(context_length, context_length, dtype=torch.bool)
        )
        self.register_buffer("causal_mask", mask)

    def forward(
        self,
        x: torch.Tensor,
        return_attention: bool = False,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        batch_size, token_count, embedding_size = x.shape
        combined = self.query_key_value(x)
        query, key, value = combined.chunk(3, dim=-1)

        def split_heads(tensor: torch.Tensor) -> torch.Tensor:
            tensor = tensor.view(
                batch_size,
                token_count,
                self.number_of_heads,
                self.head_size,
            )
            return tensor.transpose(1, 2)

        query = split_heads(query)
        key = split_heads(key)
        value = split_heads(value)

        scores = query @ key.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_size)
        visible_positions = self.causal_mask[:token_count, :token_count]
        scores = scores.masked_fill(
            ~visible_positions,
            float("-inf"),
        )
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.attention_dropout(attention_weights)

        mixed_values = attention_weights @ value
        mixed_values = mixed_values.transpose(1, 2).contiguous()
        mixed_values = mixed_values.view(
            batch_size,
            token_count,
            embedding_size,
        )
        output = self.output_projection(mixed_values)
        output = self.output_dropout(output)
        if return_attention:
            return output, attention_weights
        return output, None


class TransformerBlock(nn.Module):
    """One causal-attention layer followed by a small neural network."""

    def __init__(
        self,
        embedding_size: int,
        number_of_heads: int,
        context_length: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.first_normalization = nn.LayerNorm(embedding_size)
        self.attention = CausalSelfAttention(
            embedding_size=embedding_size,
            number_of_heads=number_of_heads,
            context_length=context_length,
            dropout=dropout,
        )
        self.second_normalization = nn.LayerNorm(embedding_size)
        self.feed_forward = nn.Sequential(
            nn.Linear(embedding_size, 4 * embedding_size),
            nn.GELU(),
            nn.Linear(4 * embedding_size, embedding_size),
            nn.Dropout(dropout),
        )

    def forward(
        self,
        x: torch.Tensor,
        return_attention: bool = False,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        attention_output, attention_weights = self.attention(
            self.first_normalization(x),
            return_attention=return_attention,
        )
        x = x + attention_output
        x = x + self.feed_forward(self.second_normalization(x))
        return x, attention_weights


class Day4TinyGPT(nn.Module):
    """A small decoder-only Transformer for next-character prediction."""

    def __init__(
        self,
        vocabulary_size: int,
        context_length: int = MODEL_CONFIG["context_length"],
        embedding_size: int = MODEL_CONFIG["embedding_size"],
        number_of_heads: int = MODEL_CONFIG["number_of_heads"],
        number_of_blocks: int = MODEL_CONFIG["number_of_blocks"],
        dropout: float = MODEL_CONFIG["dropout"],
    ) -> None:
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(
            vocabulary_size,
            embedding_size,
        )
        self.position_embedding = nn.Embedding(
            context_length,
            embedding_size,
        )
        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    embedding_size=embedding_size,
                    number_of_heads=number_of_heads,
                    context_length=context_length,
                    dropout=dropout,
                )
                for _ in range(number_of_blocks)
            ]
        )
        self.final_normalization = nn.LayerNorm(embedding_size)
        self.language_model_head = nn.Linear(
            embedding_size,
            vocabulary_size,
        )

    def forward(
        self,
        token_ids: torch.Tensor,
        targets: torch.Tensor | None = None,
        return_attention: bool = False,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor | None,
        torch.Tensor | None,
    ]:
        _, token_count = token_ids.shape
        if token_count > self.context_length:
            raise ValueError("input is longer than context_length")

        positions = torch.arange(token_count, device=token_ids.device)
        x = self.token_embedding(token_ids)
        x = x + self.position_embedding(positions)

        saved_attention = None
        for block_index, block in enumerate(self.blocks):
            should_save = return_attention and block_index == 0
            x, attention_weights = block(
                x,
                return_attention=should_save,
            )
            if should_save:
                saved_attention = attention_weights

        x = self.final_normalization(x)
        logits = self.language_model_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )
        return logits, loss, saved_attention


def count_parameters(model: nn.Module) -> int:
    """Return the number of trainable values stored inside a model."""

    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


@torch.no_grad()
def generate_text_ids(
    model: Day4TinyGPT,
    starting_ids: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
) -> torch.Tensor:
    """Generate new token IDs one at a time."""

    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")
    model.eval()
    generated = starting_ids
    for _ in range(max_new_tokens):
        model_input = generated[:, -model.context_length :]
        logits, _, _ = model(model_input)
        next_token_logits = logits[:, -1, :] / temperature
        probabilities = F.softmax(next_token_logits, dim=-1)
        next_id = torch.multinomial(probabilities, num_samples=1)
        generated = torch.cat([generated, next_id], dim=1)
    return generated


In [ ]:
MODEL_CONFIG = {
    "context_length": 64,
    "embedding_size": 128,
    "number_of_heads": 4,
    "number_of_blocks": 2,
    "dropout": 0.1,
}

torch.manual_seed(42)
model = Day4TinyGPT(
    vocabulary_size=vocabulary_size,
    **MODEL_CONFIG,
).to(device)

print("Model:", model.__class__.__name__)
print("Trainable parameters:", f"{count_parameters(model):,}")
print("Context length:", MODEL_CONFIG["context_length"])


## Create Random Training Batches

`get_batch(...)` selects short regions from the token stream:

- input: 64 consecutive token IDs;
- target: the same region shifted one position forward.

| Name or parameter | Value here | Role |
|---|---:|---|
| `get_batch(split)` | `"train"` or `"validation"` | chooses which token stream to sample |
| `batch_size` | `32` | number of short sequences processed together |
| `context_length` | `64` | token positions in each sequence |
| `torch.randint(...)` | random valid starting positions | selects different text regions |
| `torch.stack(...)` | list of equal-length tensors | combines sequences into one batch tensor |
| `.to(device)` | CPU or GPU | moves inputs and targets beside the model |

A batch shape of `(32, 64)` means 32 sequences, with 64 token
positions in each sequence.


In [ ]:
context_length = MODEL_CONFIG["context_length"]
batch_size = 32

def get_batch(split):
    source = (
        training_ids
        if split == "train"
        else validation_ids
    )
    starts = torch.randint(
        0,
        len(source) - context_length - 1,
        (batch_size,),
    )
    inputs = torch.stack(
        [
            source[start : start + context_length]
            for start in starts
        ]
    )
    targets = torch.stack(
        [
            source[start + 1 : start + context_length + 1]
            for start in starts
        ]
    )
    return inputs.to(device), targets.to(device)

test_inputs, test_targets = get_batch("train")
print("Input batch shape:", tuple(test_inputs.shape))
print("Target batch shape:", tuple(test_targets.shape))


## Generate Before Training

This first result provides a baseline. The generation function is
ready, but the model weights are still random.

| Name or parameter | Role |
|---|---|
| `generation_prompt` | starting text supplied by a person |
| `encode(...)` | converts the prompt into token IDs |
| `torch.tensor([ids], dtype=torch.long, device=device)` | creates a two-dimensional model input on the correct device |
| `model.eval()` | disables training-only dropout |
| `torch.no_grad()` | avoids storing gradients during prediction |
| `model(starting_ids)` | produces logits for every prompt position |
| `torch.softmax(..., dim=-1)` | converts the final-position logits into probabilities across the vocabulary |
| `max_new_tokens=120` | asks generation to append 120 tokens |
| `temperature=1.0` | keeps the original probability scale |

`dim=-1` means “apply softmax across the final dimension,” which is
the vocabulary dimension in this model.


In [ ]:
generation_prompt = "Question"
starting_ids = torch.tensor(
    [encode(generation_prompt)],
    dtype=torch.long,
    device=device,
)

model.eval()
with torch.no_grad():
    baseline_logits, _, _ = model(starting_ids)
    before_training_probabilities = torch.softmax(
        baseline_logits[0, -1],
        dim=-1,
    ).cpu()

torch.manual_seed(7)
untrained_ids = generate_text_ids(
    model,
    starting_ids,
    max_new_tokens=120,
    temperature=1.0,
)
untrained_text = decode(untrained_ids[0].cpu())
print(untrained_text)


# Part 6 — Train TinyGPT

<div style="overflow-x:auto;">
  <table style="width:100%; border-collapse:separate; border-spacing:7px;">
    <tr>
      <td style="background:#e8f1ff; border:2px solid #246bce; border-radius:12px; padding:14px; text-align:center;">
        <b>Training text</b><br><small>food corpus</small>
      </td>
      <td style="border:0; text-align:center; font-size:24px;">→</td>
      <td style="background:#e8f1ff; border:2px solid #246bce; border-radius:12px; padding:14px; text-align:center;">
        <b>Tokenizer</b><br><small>text → token IDs</small>
      </td>
      <td style="border:0; text-align:center; font-size:24px;">→</td>
      <td style="background:#e8f1ff; border:2px solid #246bce; border-radius:12px; padding:14px; text-align:center;">
        <b>Shift pairs</b><br><small>inputs + targets</small>
      </td>
      <td style="border:0; text-align:center; font-size:24px;">→</td>
      <td style="background:#fff2d8; border:2px solid #b96900; border-radius:12px; padding:14px; text-align:center;">
        <b>TinyGPT</b><br><small>next-token logits</small>
      </td>
      <td style="border:0; text-align:center; font-size:24px;">→</td>
      <td style="background:#fff2d8; border:2px solid #b96900; border-radius:12px; padding:14px; text-align:center;">
        <b>Loss</b><br><small>prediction vs. target</small>
      </td>
      <td style="border:0; text-align:center; font-size:24px;">→</td>
      <td style="background:#fff2d8; border:2px solid #b96900; border-radius:12px; padding:14px; text-align:center;">
        <b>Update weights</b><br><small>backpropagation + optimizer</small>
      </td>
    </tr>
  </table>
</div>

Important training functions and parameters:

| Name or parameter | Role in learning |
|---|---|
| `torch.optim.AdamW(...)` | creates the optimizer that changes parameters |
| `model.parameters()` | supplies all trainable values to the optimizer |
| `lr=3e-4` | learning rate; controls the size of each update |
| `model.train()` | enables training behavior such as dropout |
| `model(inputs, targets)` | runs a forward pass and calculates cross-entropy loss |
| `optimizer.zero_grad()` | clears gradients left from the previous step |
| `loss.backward()` | calculates how each parameter affected the loss |
| `optimizer.step()` | uses the gradients to update parameters |
| `live_training_steps` | number of repeated parameter updates |
| `report_every` | interval between displayed measurements |
| `evaluation_batches=5` | number of batches averaged for each displayed loss |

Every step uses four central operations:

```python
optimizer.zero_grad()
logits, loss, _ = model(inputs, targets)
loss.backward()
optimizer.step()
```

- loss measures next-token mistakes;
- `.backward()` calculates gradients;
- `.step()` updates the weights.

The graph checks two kinds of loss:

- **training loss** measures short batches from the text used to
  update the model;
- **validation loss** measures separate text without updating the
  model.

For each displayed point, five short batches are averaged to reduce
random movement. During validation, `model.eval()` changes the model
to evaluation mode and `torch.no_grad()` prevents gradient
calculation. Validation never calls `.backward()` or `.step()`.

After the loop, `torch.softmax(...)` converts next-token logits
into probabilities and `torch.topk(...)` selects the largest
probabilities. A side-by-side chart will show how training changed
the model’s next-token prediction after the same prompt.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
)
live_training_steps = (
    1000
    if device.type in ("cuda", "mps")
    else 500
)
report_every = max(1, live_training_steps // 10)
evaluation_batches = 5
recorded_steps = []
recorded_training_losses = []
recorded_validation_losses = []

model.train()
for step in range(1, live_training_steps + 1):
    inputs, targets = get_batch("train")
    optimizer.zero_grad()
    _, loss, _ = model(inputs, targets)
    loss.backward()
    optimizer.step()

    if step == 1 or step % report_every == 0:
        model.eval()
        training_check_losses = []
        validation_check_losses = []

        with torch.no_grad():
            for _ in range(evaluation_batches):
                check_inputs, check_targets = get_batch(
                    "train"
                )
                _, check_loss, _ = model(
                    check_inputs,
                    check_targets,
                )
                training_check_losses.append(
                    check_loss.item()
                )

                check_inputs, check_targets = get_batch(
                    "validation"
                )
                _, check_loss, _ = model(
                    check_inputs,
                    check_targets,
                )
                validation_check_losses.append(
                    check_loss.item()
                )

        average_training_loss = (
            sum(training_check_losses)
            / evaluation_batches
        )
        average_validation_loss = (
            sum(validation_check_losses)
            / evaluation_batches
        )
        recorded_steps.append(step)
        recorded_training_losses.append(
            average_training_loss
        )
        recorded_validation_losses.append(
            average_validation_loss
        )
        print(
            f"step {step:>3}/{live_training_steps} "
            f"training loss "
            f"{average_training_loss:.4f} | "
            f"validation loss "
            f"{average_validation_loss:.4f}"
        )
        model.train()

figure, axis = plt.subplots(figsize=(9, 4.5))
axis.plot(
    recorded_steps,
    recorded_training_losses,
    marker="o",
    color="#2563eb",
    label="Training loss",
)
axis.plot(
    recorded_steps,
    recorded_validation_losses,
    marker="s",
    color="#d97706",
    label="Validation loss",
)
axis.set_xlabel("Training step")
axis.set_ylabel("Next-token loss")
axis.legend()
axis.grid(alpha=0.25)
figure.tight_layout()
plt.show()

model.eval()
with torch.no_grad():
    trained_logits, _, _ = model(starting_ids)
    after_training_probabilities = torch.softmax(
        trained_logits[0, -1],
        dim=-1,
    ).cpu()

top_candidate_ids = torch.topk(
    after_training_probabilities,
    k=8,
).indices
candidate_labels = [
    repr(decode([int(token_id)]))
    for token_id in top_candidate_ids
]
before_values = before_training_probabilities[
    top_candidate_ids
].numpy()
after_values = after_training_probabilities[
    top_candidate_ids
].numpy()

x_positions = np.arange(len(candidate_labels))
bar_width = 0.38
figure, axis = plt.subplots(figsize=(10, 4.5))
axis.bar(
    x_positions - bar_width / 2,
    before_values,
    width=bar_width,
    label="Before training",
    color="#bfdbfe",
    edgecolor="#2563eb",
)
axis.bar(
    x_positions + bar_width / 2,
    after_values,
    width=bar_width,
    label="After live training",
    color="#fde68a",
    edgecolor="#d97706",
)
axis.set_xticks(x_positions)
axis.set_xticklabels(candidate_labels)
axis.set_xlabel("Possible next character after 'Question'")
axis.set_ylabel("Probability")
axis.legend()
axis.grid(axis="y", alpha=0.25)
figure.tight_layout()
plt.show()


# Part 7 — Load the Longer-Trained Model

A **checkpoint** stores trained parameter values. The prepared
checkpoint uses the same:

- tokenizer vocabulary;
- TinyGPT structure;
- food-text corpus.

`torch.load(...)` reads the saved checkpoint.
`load_state_dict(...)` copies its trained weights into a matching
model object.

| Name or parameter | Role |
|---|---|
| `urlretrieve(url, path)` | downloads the checkpoint when no local copy exists |
| `torch.load(checkpoint_path, ...)` | reads the saved Python/PyTorch data |
| `map_location=device` | loads tensors for the current CPU or GPU |
| `weights_only=True` | restricts loading to safe weight-related data |
| `checkpoint["model_config"]` | retrieves the architecture settings used during training |
| `checkpoint["model_state_dict"]` | retrieves the learned parameter tensors |
| `prepared_model.load_state_dict(...)` | copies saved tensors into the matching model |
| `prepared_model.eval()` | prepares the loaded model for generation |

A checkpoint does not contain a magical new algorithm. It contains
the configuration and parameter values produced by earlier
training.


In [ ]:
from urllib.request import urlretrieve

CHECKPOINT_URL = "https://raw.githubusercontent.com/cook1e-0707/practicalAI-lab/main/day4/model/day4_tinygpt_checkpoint.pt"
local_checkpoint_candidates = [
    Path("day4/model/day4_tinygpt_checkpoint.pt"),
    Path("../model/day4_tinygpt_checkpoint.pt"),
    Path("../../day4/model/day4_tinygpt_checkpoint.pt"),
]
local_checkpoint_path = next(
    (
        path
        for path in local_checkpoint_candidates
        if path.exists()
    ),
    None,
)

if local_checkpoint_path is not None:
    checkpoint_path = local_checkpoint_path
    checkpoint_source = (
        f"local file: {checkpoint_path.resolve()}"
    )
else:
    checkpoint_path = Path("/content/day4_tinygpt_checkpoint.pt")
    if not Path("/content").exists():
        checkpoint_path = Path("day4_tinygpt_checkpoint.pt")
    urlretrieve(CHECKPOINT_URL, checkpoint_path)
    checkpoint_source = "GitHub raw-data URL"

try:
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True,
    )
except TypeError:
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

prepared_model = Day4TinyGPT(
    vocabulary_size=vocabulary_size,
    **checkpoint["model_config"],
).to(device)
prepared_model.load_state_dict(
    checkpoint["model_state_dict"]
)
prepared_model.eval()
prepared_checkpoint_loaded = True

print("Checkpoint source:", checkpoint_source)
print("Training steps:", checkpoint["training_steps"])
print(
    "Validation loss:",
    round(checkpoint["validation_loss"], 4),
)


## Generate with Trained Weights

<div style="overflow-x:auto;">
  <table style="width:100%; border-collapse:separate; border-spacing:7px;">
    <tr>
      <td style="background:#e5f7f2; border:2px solid #138a72; border-radius:12px; padding:14px; text-align:center;">
        <b>Prompt</b><br><small>starting text → token IDs</small>
      </td>
      <td style="border:0; text-align:center; font-size:24px;">→</td>
      <td style="background:#e5f7f2; border:2px solid #138a72; border-radius:12px; padding:14px; text-align:center;">
        <b>Trained TinyGPT</b><br><small>use learned weights</small>
      </td>
      <td style="border:0; text-align:center; font-size:24px;">→</td>
      <td style="background:#e5f7f2; border:2px solid #138a72; border-radius:12px; padding:14px; text-align:center;">
        <b>Next token</b><br><small>logits → probability</small>
      </td>
      <td style="border:0; text-align:center; font-size:24px;">→</td>
      <td style="background:#e5f7f2; border:2px solid #138a72; border-radius:12px; padding:14px; text-align:center;">
        <b>Append + repeat</b><br><small>add one token and predict again</small>
      </td>
      <td style="border:0; text-align:center; font-size:24px;">→</td>
      <td style="background:#e5f7f2; border:2px solid #138a72; border-radius:12px; padding:14px; text-align:center;">
        <b>Generated text</b><br><small>decode token IDs</small>
      </td>
    </tr>
  </table>
</div>

Important generation operations:

| Name or parameter | Role |
|---|---|
| `prepared_model.eval()` | disables dropout during generation |
| `torch.no_grad()` | prevents gradient storage because weights are not changing |
| `generated[:, -context_length:]` | keeps only the most recent tokens the model can use |
| `logits[:, -1, :]` | selects vocabulary logits for the final input position |
| `temperature` | changes how concentrated or varied the probabilities are |
| `torch.softmax(..., dim=-1)` | converts vocabulary logits into probabilities |
| `torch.multinomial(..., num_samples=1)` | samples one next-token ID |
| `torch.cat(..., dim=1)` | appends the sampled ID to the token sequence |
| `max_new_tokens` | number of times the predict-and-append loop repeats |
| `decode(...)` | converts the completed ID sequence back into text |

During generation, the model parameters remain fixed. The growing
token sequence changes, but the weights do not.


In [ ]:
visual_prompt = "Question"
visual_generated_ids = torch.tensor(
    [encode(visual_prompt)],
    dtype=torch.long,
    device=device,
)
visual_generation_rows = []

torch.manual_seed(12)
prepared_model.eval()
with torch.no_grad():
    for step in range(1, 6):
        text_before = decode(
            visual_generated_ids[0].cpu()
        )
        model_input = visual_generated_ids[
            :,
            -prepared_model.context_length :,
        ]
        logits, _, _ = prepared_model(model_input)
        next_token_logits = logits[:, -1, :] / 0.8
        probabilities = torch.softmax(
            next_token_logits,
            dim=-1,
        )
        next_id = torch.multinomial(
            probabilities,
            num_samples=1,
        )
        next_character = decode(
            [int(next_id.item())]
        )
        visual_generated_ids = torch.cat(
            [visual_generated_ids, next_id],
            dim=1,
        )
        text_after = decode(
            visual_generated_ids[0].cpu()
        )
        visual_generation_rows.append(
            [
                step,
                text_before.replace("\n", "↵"),
                repr(next_character),
                int(next_id.item()),
                text_after.replace("\n", "↵"),
            ]
        )

figure, axis = plt.subplots(figsize=(13, 4.2))
axis.axis("off")
generation_table = axis.table(
    cellText=visual_generation_rows,
    colLabels=[
        "Step",
        "Text entering TinyGPT",
        "Sampled token",
        "Token ID",
        "Text after append",
    ],
    cellLoc="center",
    loc="center",
    colWidths=[0.07, 0.28, 0.14, 0.10, 0.28],
)
generation_table.auto_set_font_size(False)
generation_table.set_fontsize(10)
generation_table.scale(1, 1.8)

for column_index in range(5):
    generation_table[(0, column_index)].set_facecolor(
        "#dbeafe"
    )
for row_index in range(1, len(visual_generation_rows) + 1):
    generation_table[(row_index, 2)].set_facecolor(
        "#fef3c7"
    )
    generation_table[(row_index, 4)].set_facecolor(
        "#dcfce7"
    )

figure.tight_layout()
plt.show()


In [ ]:
prepared_prompt = (
    "Question: Which foods provide vitamin C?\n"
    "Verified answer:"
)
prepared_start = torch.tensor(
    [encode(prepared_prompt)],
    dtype=torch.long,
    device=device,
)

torch.manual_seed(12)
prepared_output_ids = generate_text_ids(
    prepared_model,
    prepared_start,
    max_new_tokens=220,
    temperature=0.8,
)
prepared_text = decode(prepared_output_ids[0].cpu())
print(prepared_text)


## Task 2 — Generate from Your Prompt

**Input:** a food-related prompt using characters in the vocabulary.

**Expected output:** the prompt followed by generated text.

You will use the same four-step generation interface:

| Step | Function or parameter |
|---:|---|
| 1 | `encode(your_prompt)` converts text into IDs |
| 2 | `torch.tensor([ids], dtype=torch.long, device=device)` makes the model input |
| 3 | `generate_text_ids(prepared_model, custom_start, max_new_tokens=160, temperature=0.8)` generates IDs |
| 4 | `decode(custom_ids[0].cpu())` converts the first generated sequence into text |

`max_new_tokens` controls output length. `temperature` controls
sampling variety. Neither parameter changes the trained weights.

<details><summary>Hint 1</summary>

Encode the prompt inside a two-dimensional `torch.long` tensor.
</details>

<details><summary>Hint 2</summary>

```python
custom_start = torch.tensor(
    [encode(your_prompt)],
    dtype=torch.long,
    device=device,
)
custom_ids = generate_text_ids(
    prepared_model,
    custom_start,
    max_new_tokens=160,
    temperature=0.8,
)
```
</details>


In [ ]:
your_prompt = "Question: What foods contain calcium?\nVerified answer:"
custom_start = None  # TODO: encode the prompt in a tensor
custom_ids = None  # TODO: call generate_text_ids(...)
custom_text = None  # TODO: decode custom_ids
print(custom_text)


# Final Review

```text
text
  → tokenizer
  → token IDs
  → shifted inputs and targets
  → TinyGPT predictions
  → loss
  → backpropagation and weight updates
  → trained next-token generation
```

1. What is the difference between a tokenizer and tokenization?
2. Why is the target shifted by one token?
3. What does loss measure?
4. What changes when `optimizer.step()` runs?
5. How does one predicted token become a longer text?

Generated text can resemble the training corpus without being a
reliable factual answer.
